# <center>Claude Code 专题课第 4 节：从 Coordinator 到 Agent Teams</center>

&emsp;&emsp;上一节课我们用三章篇幅拆透了 `Claude Code` 多 Agent 的三块基石——子 Agent 隔离、Fork 缓存共享、Coordinator 编排——并在第九章用「四对张力」把整个架构收口成一把可以量任何 Agent 系统的尺。其中「集中 vs 分治」这对张力留下了一个未解的问号：Coordinator 把所有编排决策集中到自己身上，子 Agent 只管执行完就走——这套 hub-and-spoke 架构在什么场景下会碰到天花板？碰到之后怎么办？

&emsp;&emsp;今天这节课就是来回答这个问题的。我们会打开 `Claude Code` 在 2026 年上半年引入的一套实验性机制——**Agent Teams**。它不是 Coordinator 的替代品，而是在 Coordinator 之上叠加的一层 P2P 通信与共享任务协调能力。你会看到 6 个新工具、一个文件系统邮箱、一套共享 TaskList、三种后端类型和一套身份系统，这些零件拼在一起，让 Agent 之间从「派活-收活」的单向指令升级到了「持久化驻留 + 双向协商 + 自主认领」的团队协作模式。

&emsp;&emsp;为了把这条主线讲透，我们按五章递进展开：先用第一章建立「旧模式为什么不够用」的认知动力，然后在第二章打开通信层的两个核心机制（SendMessage P2P 寻址和 Mailbox 异步投递），第三章深入任务协调层（共享 TaskList 与 `blockedBy` 依赖链），第四章落到执行基础设施（三种后端 + 身份系统 + 生命周期），最后第五章回到四对张力做认知升级，同时讲清 Agent Teams 的启用机制和实验性功能边界。每一块机制都会落到可以 `grep` 复核的源码锚点——这是本系列课程一贯的纪律。

> 📌 **目标受众与前置要求**：本课面向已经完成第三节课第 5-7 章的学员——你需要能讲清子 Agent 隔离（`AgentTool.tsx`）、Fork 缓存共享（`CacheSafeParams`）和 Coordinator 编排（`coordinatorMode.ts`）的核心原理，并熟悉四对张力框架。技术上你需要能读 TypeScript 源码片段，**不需要**搭建 Agent Teams 运行环境（这是实验性功能，feature flag 门控，无法在标准环境复现）。

> 📌 **学完本节你将带走 5 件产物**：① 能判断何时 Coordinator 模式不够用、需要升级到 Agent Teams；② 能从源码层面解释 6 个工具的职能分工和 Mailbox 异步投递设计；③ 能描述 Task 数据模型、`blockedBy` 依赖链和「隐式 vs 显式」任务管理的结构差异；④ 能解释三种后端类型和身份系统的工作原理；⑤ 能将第三节课四对张力框架映射到 Agent Teams，理解「有 leader 的 P2P」这个核心设计取舍。

> 📅 **时效性说明**：本课全部源码引用基于 `Claude Code` 本地快照（2026 年 5 月下旬验证）。Agent Teams 是实验性功能（`experimental`），API 和行为可能随版本变化。所有 `file:line` 引用都是真实可核对的——你可以在本地 `grep` 对应文件验证。

---

## 0. 动手前：如何启用 Agent Teams

&emsp;&emsp;Agent Teams 是 `Claude Code` 的**实验性功能**，默认关闭。在开始阅读架构分析之前，我们先把启用方式讲清楚——你可以选择现在就开启体验，也可以只读源码理解设计，两种路径都不影响后续章节的学习。

### 0.1 启用方式

&emsp;&emsp;对外部用户（非 Anthropic 内部员工），**唯一可靠的启用方式是设置环境变量**。编辑 `~/.claude/settings.json`，在 `env` 字段中添加：

```json
{
  "env": {
    "CLAUDE_CODE_EXPERIMENTAL_AGENT_TEAMS": "1"
  }
}
```

&emsp;&emsp;`settings.json` 的 `env` 字段会在 Claude Code 启动时通过 `Object.assign(process.env, ...)` 注入到 `process.env`（`src/utils/managedEnv.ts:136`），效果等同于在终端里 `export` 环境变量，但**跨 session 持久化**——你不需要每次启动都重新设。

> **【关于 `--agent-teams` CLI flag】**：源码 `agentSwarmsEnabled.ts:11` 有一个 `isAgentTeamsFlagSet()` 函数，检查 `process.argv.includes('--agent-teams')`。但这个 flag **只在 Anthropic 内部构建版本（`ant`）的 CLI 参数解析器中注册**——外部公开版本的 CLI parser 遇到这个未注册的 flag 会直接报 `error: unknown option '--agent-teams'`，代码根本走不到 `process.argv` 检查那一步。源码注释说"if external users pass it anyway, it will work"是理想情况，实际上 CLI parser 先拦住了。**结论：外部用户不要用这个 flag，直接用 `settings.json` 环境变量方式。**

### 0.2 验证是否启用成功

&emsp;&emsp;启用后，你可以用以下方式验证 Agent Teams 是否真正生效：

&emsp;&emsp;**验证一：检查 `TeamCreate` 工具是否可用**。在 Claude Code 对话中输入"创建一个 team"或直接要求使用 `TeamCreate` 工具。如果 Agent Teams 已启用，Claude Code 会调用 `TeamCreateTool`，在 `~/.claude/teams/` 下创建 team 配置目录；如果未启用，`TeamCreate` 工具的 `isEnabled()` 返回 `false`（`TeamCreateTool.ts:89`），工具不会出现在可用列表中。

&emsp;&emsp;**验证二：检查 `config.json` 是否生成**。team 创建成功后，检查对应的配置文件：

```bash
# 查看已创建的 team
ls ~/.claude/teams/

# 查看某个 team 的配置
cat ~/.claude/teams/{team-name}/config.json

# 查看共享 TaskList 目录
ls ~/.claude/tasks/{team-name}/
```

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260526201544397.png" width=50%></div>

&emsp;&emsp;如果 `config.json` 存在且包含 `leadAgentId` 和 `members` 数组，说明 Agent Teams 已成功启用并创建了 team。

&emsp;&emsp;**验证三（可选）：观察 tmux 后端的可视化效果**。如果你碰巧在 tmux 环境中运行 Claude Code（`brew install tmux` + `tmux new-session -s claude`），创建 team 后会看到新的 tmux pane 被自动创建——每个 teammate 一个独立 pane。这是最直观的"启用成功"信号，但**不是必须的**——不装 tmux 也能用 Agent Teams，系统会自动 fallback 到 in-process 后端（第四章会详细讲三种后端的选择逻辑）。

### 0.3 注意事项

> **【关于 GrowthBook killswitch】**：即使你手动启用了环境变量或 CLI flag，Anthropic 仍然可以通过远程 GrowthBook 配置（`tengu_amber_flint` killswitch，`agentSwarmsEnabled.ts:39`）关闭这个功能。如果启用后仍然找不到 `TeamCreate` 工具，可能是 killswitch 处于关闭状态——这不是你的配置问题，而是 Anthropic 的远程控制。

> **【关于 "Backgrounded agent" 提示】**：在 Claude Code 中看到 `Backgrounded agent (↓ to manage · ctrl+o to expand)` 的提示**不代表 Agent Teams 已启用**——那是普通子 Agent 的 `run_in_background` 后台运行模式（`AgentTool/UI.tsx:348`），属于第三节课讲的 Coordinator dispatch 体系。Agent Teams 的标志是 `~/.claude/teams/` 下出现了 `config.json`，以及 tmux/iTerm2 中出现了独立的 teammate pane。

### 0.4 实战演练：在 tmux 中体验 Agent Teams

&emsp;&emsp;以下是一次真实的 Agent Teams 操作过程记录。你可以跟着步骤操作，也可以只读这一节建立直觉——后面的架构分析不依赖这次实操。

**步骤一：启动 tmux 并进入 Claude Code**

```bash
# 安装 tmux（如未安装）
brew install tmux

# 创建一个 tmux session
tmux new-session -s claude1

# 在 tmux 里启动 Claude Code
claude
```

&emsp;&emsp;此时你的终端只有一个 pane——team lead 的主对话窗口。

**步骤二：创建 team 并观察 pane 自动分裂**

&emsp;&emsp;在 Claude Code 对话中输入：

In [ ]:
创建一个 team，名字叫 test-team，spawn 两个 teammate：
一个叫 researcher 负责搜索和调研，一个叫 writer 负责写作

&emsp;&emsp;执行后你会看到 tmux 窗口**自动分裂成三个 pane**——左侧是 team lead，右侧上方是 researcher，右侧下方是 writer。每个 pane 各自运行一个完整的 Claude Code 实例，互相独立。两个 teammate 会依次"报到"：`@researcher Researcher reporting for duty`、`@writer Writer reporting for duty`。下面这张截图是实际运行效果：

&emsp;&emsp;截图中可以看到：左侧 pane 是 team lead，它显示了 team 的创建过程和两个 teammate 的报到信息；右侧上方是 researcher（标注 `general-purpose`），右侧下方是 writer（标注 `@writer`）——三个 pane 各自有独立的上下文信息栏（模型、token 用量、MCP 连接数等），说明每个 teammate 确实是一个完整的 Claude Code 实例。底部状态栏显示 `Team test-team, 2 teammates`，两个 teammate 都处于 `[idle]` 状态等待任务分配。

**步骤三：在 pane 之间切换与常用 tmux 操作**

&emsp;&emsp;创建 team 后，tmux 把一个 window 分裂成多个 pane——team lead 一个，每个 teammate 各一个。你需要在这些 pane 之间来回切换查看状态，有时还要回滚某个 teammate 刷过去的历史输出、或临时离开再重连。下面这张速查表覆盖了这些场景的核心快捷键（所有快捷键都是**先按 `ctrl+b` 松开，再按第二个键**）：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 0-1 tmux 实操速查</font></p>
<div class="center">

| 类别 | 快捷键 | 功能 |
|------|--------|------|
| **pane 导航** | `ctrl+b` 然后 `o` | 切换到下一个 pane |
| | `ctrl+b` 然后 `←` `→` `↑` `↓` | 按方向切换到指定 pane |
| | `ctrl+b` 然后 `q` | 短暂显示每个 pane 的编号，趁编号未消失按数字键跳转——teammate 一多时这是最快的定位方式 |
| | `ctrl+b` 然后 `z` | 放大 / 还原当前 pane（全屏查看某个 teammate 的输出） |
| **历史回滚** | `ctrl+b` 然后 `[` | 进入复制模式，用 `↑` `↓` / `PageUp` `PageDown` 回看已经刷过去的输出，按 `q` 退出 |
| **窗口 / 会话** | `ctrl+b` 然后 `w` | 交互式列出所有 window，用方向键选择后回车跳转 |
| | `ctrl+b` 然后 `n` / `p` | 切换到下一个 / 上一个 window |
| | `ctrl+b` 然后 `d` | detach 脱离当前 session（session 在后台继续运行，之后 `tmux attach` 可重连，详见 0.7 节） |

</div>

**步骤四：给 team lead 下达任务——注意"口头派活"的陷阱**

&emsp;&emsp;你可能会本能地在 team lead 的 pane 里直接说"让 researcher 去搜 XXX"。team lead 会回复"已派活"——但切到 researcher 的 pane，你会发现它说**"仍然没有收到任务分配"**。

&emsp;&emsp;这是 Agent Teams 当前的一个**时序问题**：teammate 报到后进入 idle 状态（Stop hook 触发了 `createIdleNotification`），而 **idle 状态的 teammate 不会主动轮询收件箱**。team lead 通过 `SendMessage` 发出的消息躺在收件箱文件里，但没有机制唤醒已经 idle 的 teammate 去读它。<font color=red>这恰好印证了第五章会讲的"P2P 局限三：没有 watchdog 机制"</font>。

**步骤五：两种有效的任务下达方式**

&emsp;&emsp;**方式 A（推荐）：通过 TaskCreate 创建结构化任务**。在 team lead 的 pane 里输入：

In [ ]:
帮我用 TaskCreate 创建两个任务：
1. subject="搜索 Claude Code 最新功能"，description="搜索并整理 Claude Code 2026 年的功能更新"
2. subject="撰写功能总结"，description="基于 researcher 的搜索结果撰写总结"，blockedBy 第一个任务
然后用 SendMessage 通知 researcher 去查看 TaskList 认领任务

&emsp;&emsp;这走的是第三章会讲的"显式 TaskList"正式流程——任务以结构化数据存储在 `~/.claude/tasks/test-team/` 下（含 `.lock` 锁文件和 `.highwatermark` 水位标记），任何 teammate 调用 `TaskList` 都能看到并认领。注意 task 数据和 teammate 之间的**消息中间交流文件**是分开存储的——后者在 `~/.claude/teams/test-team/inboxes/{agent_name}.json`，由 `SendMessage` 写入、由接收方轮询拉取（详见第二章 Mailbox 设计与第三章 TaskList 结构）。

&emsp;&emsp;**方式 B：直接在 teammate 的 pane 里对话**。切到 researcher 的 pane（`ctrl+b` 然后 `→`），直接输入：

In [ ]:
搜索一下 Claude Code 2026 年的最新功能更新，整理成要点列表

&emsp;&emsp;因为每个 teammate 都是一个**完整的 Claude Code 实例**，你直接跟它对话就等于"唤醒"了它。它拥有完整的工具集——`Read` / `Write` / `Bash` / `Grep` / `WebSearch` / `Skill` / `Agent`（可以再 spawn 子 Agent）全部可用。完成后它可以用 `SendMessage(to="writer", ...)` 把结果直接发给 writer。

> **【关键认知】**：Agent Teams 里的 teammate **不是**能力受限的 worker。第三节课讲的 Coordinator 模式下，worker 只有 `Bash` / `Read` / `Edit` 三个基础工具。但 teammate 是完整的 `claude` 进程——它能调用本地 `skills`、能 `spawn` 子 Agent、能访问 `MCP` 服务器、能读写文件。唯一的额外约束是系统提示词里追加了一段话（`teammatePromptAddendum.ts`），告诉它"你在一个 team 里，跟别人沟通必须用 `SendMessage`，光写文本对方看不到"。这是**行为引导**，不是能力裁减。

&emsp;&emsp;下面这张截图展示了 teammate 之间成功协作的完整效果：

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260526201553537.png" width=50%></div>

&emsp;&emsp;截图中可以看到完整的协作链路：左侧 team lead 下达任务后，右上方的 researcher 完成了调研并产出了 `claude-code-2026-updates.md`（175 行，覆盖 7 个章节），然后 researcher 主动验证了 writer 的产出——确认文件结构完整、内容覆盖六大板块、技术风格符合要求。右下方的 writer 完成写作后收到了 researcher 的验证确认。<font color=red>注意这里的关键点：researcher 和 writer 之间的交互（发送结果、验证产出）是通过 `SendMessage` P2P 直接完成的，没有经过 team lead 中转</font>——这正是 Agent Teams 与 Coordinator 模式最本质的区别。

**步骤六：清理**

```bash
# 在 team lead 的 pane 里说"删除 test-team"
# 或手动清理：
# 前者含 config.json 与 inboxes/（mailbox 消息），后者含 TaskList 数据（.lock / .highwatermark）
rm -rf ~/.claude/teams/test-team ~/.claude/tasks/test-team

# 退出 tmux：
# ctrl+b 然后 d（detach 分离）
# tmux kill-session -t claude1（彻底关闭）
```

> **【这次实操的教学价值】**：这六个步骤里藏着 Agent Teams 的三个核心设计特征——**pane 自动分裂**证明了每个 teammate 是独立进程（第四章的 tmux 后端）；**idle 后收不到消息**暴露了文件系统邮箱的 watchdog 缺失（第五章的 P2P 局限）；**teammate 能调用完整工具集**说明了它不是受限 worker 而是完整 Claude Code 实例（第二章的架构分层——teammate 在 Coordinator 之上，不在之下）。后面四章的架构分析，你都可以回指这次实操的具体体验来建立直觉。

### 0.5 实战演练：在 iTerm2 中体验 Agent Teams

&emsp;&emsp;如果你是 macOS 用户且日常使用 iTerm2，可以体验另一种后端——iTerm2 原生 split pane。与 tmux 相比，iTerm2 后端不需要学习 tmux 快捷键，pane 分裂直接在你熟悉的 iTerm2 窗口里发生。这一节的操作步骤与 0.4 平行，你可以选择其中一种方式体验。

**前置准备一：安装 it2 CLI 工具**

&emsp;&emsp;iTerm2 后端依赖一个叫 `it2` 的命令行工具来创建和管理 split pane。注意 PyPI 上的包名是 **`it2`**（不是 `iterm2`）——这是最容易踩的坑：

```bash
# 正确——安装 it2 CLI 工具（提供 it2 可执行文件）
uv tool install it2

# 常见错误——iterm2 包只是 Python API 库，不提供 CLI
# pip install iterm2  ← 装完后 which it2 找不到任何东西
```

&emsp;&emsp;源码 `it2Setup.ts:100-102` 明确写的是 `uv tool install it2`，安装完成后验证：`it2 --version` 应该返回版本号（如 `0.2.3`）。

**前置准备二：在 iTerm2 中启用 Python API**

&emsp;&emsp;打开 iTerm2，进入 `Settings → General → Magic → Enable Python API`。这一步是**硬性前提**——源码 `it2Setup.ts:200-204` 明确给出了这条路径。如果不启用，`it2` 命令本身能运行，但 `it2 session split`（创建 pane 的核心操作）会静默失败。

&emsp;&emsp;验证方式是 `it2 session list`（不是 `it2 --version`）。为什么？源码 `detection.ts:112-119` 的注释解释了原因：

```typescript
// src/utils/swarm/backends/detection.ts:112-119
// 用 'session list' 而不是 '--version' 来检测
// 因为 --version 在 Python API 未启用时也能通过
// 但实际的 'session split' 操作会失败
export async function isIt2CliAvailable(): Promise<boolean> {
  const result = await execFileNoThrow(IT2_COMMAND, ['session', 'list'])
  return result.code === 0
}
```

&emsp;&emsp;`--version` 只检查 `it2` 二进制是否存在，而 `session list` 会真正尝试与 iTerm2 的 Python API 建立连接——如果连接失败，说明 Python API 没启用或 iTerm2 没在运行，后续的 `session split` 必然也会失败。这是一个「用真实操作做健康检查」的工程模式。

**后端检测优先级链**

&emsp;&emsp;在你启动 Claude Code 并触发 Agent Teams 时，系统会自动检测应该使用哪种后端。检测逻辑在 `registry.ts:136-231`，是一条 4 级优先级链：

```typescript
// src/utils/swarm/backends/registry.ts:136-231（简化）
// 检测优先级：tmux > iTerm2 + it2 > tmux fallback > 报错
async function detectAndGetBackend() {
  // Priority 1: 在 tmux 内 → 永远用 tmux（即使在 iTerm2 里）
  if (await isInsideTmux()) return createTmuxBackend()

  // Priority 2: 在 iTerm2 内
  if (isInITerm2()) {
    if (await isIt2CliAvailable()) return createITermBackend()  // 原生 pane
    if (await isTmuxAvailable()) return createTmuxBackend()     // tmux 兜底
    throw new Error('iTerm2 detected but no it2 CLI and no tmux')
  }

  // Priority 3: 不在 tmux 也不在 iTerm2 → 外部 tmux session
  if (await isTmuxAvailable()) return createTmuxBackend()

  // Priority 4: 什么都没有 → 报错
  throw new Error('Install tmux: brew install tmux')
}
```

&emsp;&emsp;这条优先级链有一个容易让人困惑的设计：**即使你在 iTerm2 里，只要你先进了 tmux，就永远走 tmux 后端**（Priority 1）。这意味着如果你想体验 iTerm2 原生 pane，必须**直接在 iTerm2 的 shell 里启动 `claude`**，不能先 `tmux new-session` 再启动。源码用 `TMUX` 环境变量（在模块加载时捕获）来做这个判断——一旦检测到 `TMUX` 有值，后面的 iTerm2 检测直接跳过。

**步骤一：在 iTerm2 中直接启动 Claude Code**

```bash
# 1. 打开 iTerm2（确保不在 tmux 内！）
# 2. 验证 it2 能连接 iTerm2
it2 session list    # 应返回当前 session 列表

# 3. 启动 Claude Code
claude
```

&emsp;&emsp;注意第一行的提醒——如果你之前跟着 0.4 操作过，可能还在 tmux session 里。请先 `exit` 退出 tmux，回到 iTerm2 的原生 shell，再启动 `claude`。

**步骤二：创建 team 并观察 iTerm2 原生 pane 分裂**

&emsp;&emsp;在 Claude Code 对话中输入与 0.4 相同的指令：

In [ ]:
创建一个 team，名字叫 test-team，spawn 两个 teammate：
一个叫 researcher 负责搜索和调研，一个叫 writer 负责写作

&emsp;&emsp;执行后 iTerm2 窗口会**自动分裂成原生 split pane**——布局策略与 tmux 相同（leader 左侧，teammate 右侧垂直堆叠），但视觉体验不同：没有 tmux 的绿色状态栏和分割线，而是 iTerm2 原生的 pane 分隔。下面这张截图展示了实际效果：

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260527111410748.png" width=50%></div>

&emsp;&emsp;与 0.4 的 tmux 截图对比，你会注意到两个视觉差异：iTerm2 的 pane 边界更简洁（没有 tmux 的绿色状态栏），pane 之间的切换可以直接用鼠标点击或 `⌘+⌥+方向键`（不需要记 tmux 的 `ctrl+b` 前缀）。功能上完全等价——每个 pane 仍然是一个独立的 Claude Code 进程。

**步骤三：iTerm2 后端的工程取舍**

&emsp;&emsp;iTerm2 后端在源码层面有几个值得注意的设计决策（`ITermBackend.ts`）：

&emsp;&emsp;**第一，并发锁防止 pane 创建竞争**。多个 teammate 同时 spawn 时，`acquirePaneCreationLock()` 确保 pane 创建操作串行执行——因为 `it2 session split` 依赖「当前最后一个 pane」来决定分裂方向，并发会导致布局混乱。

&emsp;&emsp;**第二，dead session 自动剪枝**。如果用户手动关闭了某个 teammate 的 pane（`⌘+W`），下次 spawn 新 teammate 时 split 会失败——`ITermBackend` 会自动检测目标 session 是否存活，剪掉死 session 后重试（`ITermBackend.ts:180-204`）。

&emsp;&emsp;**第三，性能妥协：颜色和标题设为 no-op**。每次 `it2` 调用都会 spawn 一个 Python 进程与 iTerm2 通信，开销较大。因此 `setPaneBorderColor()` 和 `setPaneTitle()` 被实现为空函数（`ITermBackend.ts:270-289`），牺牲了 UI 美化换取 spawn 速度。

> **【iTerm2 vs tmux 后端选择建议】**：如果你日常用 iTerm2 且不想学 tmux 快捷键，iTerm2 后端是更顺手的选择——鼠标点击切 pane、`⌘+⌥+方向键` 导航。如果你需要在远程服务器或 CI 环境使用 Agent Teams，tmux 是唯一选择（iTerm2 后端依赖 macOS 原生 GUI）。两者在功能上完全等价——通信机制（SendMessage + Mailbox）、协调机制（TaskList + blockedBy）、身份系统全部相同，差异仅在终端模拟器层。

### 0.6 补充说明：in-process 模式的 UI 表现

&emsp;&emsp;除了 tmux 和 iTerm2 这两种「一个 teammate 一个 pane」的后端，Agent Teams 还有第三种运行方式：**in-process 模式**——所有 teammate 跑在同一个 Node.js 进程里，没有独立的终端窗口。这意味着你**看不到**多个 pane 同时滚屏的效果。

&emsp;&emsp;in-process 模式下，UI 的表现是「时间切换」而非「空间并列」：主界面的 Spinner 区域会显示一棵 `TeammateSpinnerTree`（`src/components/Spinner/TeammateSpinnerTree.tsx`），列出每个 teammate 的名字、运行状态和 token 消耗；你可以通过快捷键切换查看某个 teammate 的工具调用详情（`InProcessTeammateDetailDialog`）；顶部 Banner 会显示当前正在查看的 teammate 名字。用一句话概括差异：**tmux/iTerm2 是空间并列**（多个 pane 同时可见），**in-process 是时间切换**（同一个窗口轮流查看不同 teammate）。

&emsp;&emsp;in-process 模式在三种情况下被激活：一是在 `settings.json` 中显式设置 `"teammateMode": "in-process"`；二是使用非交互模式（`-p` flag）时自动启用——因为没有终端 UI 来显示 pane（`registry.ts:352-358`）；三是既不在 tmux/iTerm2 中、系统也没装 tmux 时，作为最终 fallback 启用。第四章会从源码层面讲解 in-process 模式如何通过 `AsyncLocalStorage` 实现上下文隔离——这是它与 tmux/iTerm2 后端在工程实现上最本质的差异。

### 0.7 跨平台兼容性与 session 管理

&emsp;&emsp;前面 0.4-0.6 三节讲了三种后端在 macOS 上的实操体验，但学员的实际环境可能是 Linux 或 Windows，也可能在 session 之间反复切换。这一节集中处理两类实操问题：tmux session 的连接与删除，以及不同操作系统的兼容性边界——前者关系到你能不能回到之前的工作状态，后者决定了你能选哪些后端。

> **【关于本节代码块格式】**：以下命令均在**终端外部执行**（macOS Terminal / Linux shell / Windows PowerShell 或 WSL shell），**不在 Jupyter Notebook 内运行**，因此保留 markdown 代码块格式以便复制——你可以选中代码块文本直接粘贴到终端。

**tmux session 的列出与重连**

&emsp;&emsp;tmux session 一旦创建会持续运行在后台，即使你关闭终端窗口或按 `ctrl+b` 然后 `d`（detach）主动断开，session 仍然存活——所有 pane 和 Claude Code 进程都还在。需要重连时用以下命令：

```bash
# 列出当前 tmux server 管理的所有 session
tmux ls
# 等价的全称：tmux list-sessions

# 连接最近一个 session
tmux a
# 等价的全称：tmux attach-session

# 连接指定名字的 session（之前创建的 claude1）
tmux a -t claude1
```

&emsp;&emsp;`tmux ls` 是 `list-sessions` 的官方别名（man tmux 第 list-sessions 节明确标注 `alias: ls`）。`tmux a` 中的 `a` 是 tmux 接受的**唯一前缀缩写**——man 文档为 `attach-session` 标注的官方别名是 `attach`，但因为 tmux 命令解析允许唯一前缀匹配，`a` 也可以被识别为 `attach-session`。`tmux a` 的目标 session 必须已存在——detach 不会销毁 session，所以你随时可以回去。在 session 内按 `ctrl+b` 然后 `d` 就会触发 detach，断开终端连接但保留 session 在后台运行。

**tmux session 的删除**

&emsp;&emsp;实操结束后清理 session 有几种粒度可选：删指定一个、批量删其他、或彻底关掉整个 tmux server。

```bash
# 删除指定 session（按名字）
tmux kill-session -t claude1

# 删除所有其他 session，仅保留 -t 指定的 claude1
tmux kill-session -a -t claude1

# 杀掉整个 tmux server（所有 session 一起销毁，最彻底）
tmux kill-server
```

&emsp;&emsp;`kill-session -t <name>` 销毁指定 session 并关闭它管理的所有 pane。`-a` 标志是反向选择——"删除除了 `-t` 指定的之外的所有"（man tmux 原文：`If -a is given, all sessions but the specified one is killed`）。最彻底的清理是 `kill-server`，它会终止整个 tmux 后台进程和所有 session，包括 Agent Teams 创建的外部 swarm session。

> **【踩坑预警】**：`tmux kill-server` 是**不可逆的破坏性操作**——它会一次性销毁该 socket 下所有 session（包括其他正在运行的项目 session、未保存的 vim 编辑状态、运行中的服务进程），没有确认提示。执行前请用 `tmux ls` 确认当前 server 下有哪些 session，只想清理 Agent Teams 残留时优先用 `kill-session -t <name>` 精确删除。

**Agent Teams 外部 swarm session 的特殊性**

&emsp;&emsp;有一个容易忽略的细节：如果你在 iTerm2/Warp/Terminal 里**直接**启动 Claude Code（不在 tmux 内），Agent Teams 会创建一个**独立 socket 上的 tmux session**——这个 session 用普通的 `tmux ls` 看不到。源码 `constants.ts:7-14` 定义了这个 socket 的命名规则：

```typescript
// src/utils/swarm/constants.ts:12-14（jsdoc 注释在 7-11 行）
// 为外部 swarm session 生成独立 socket 名
// 包含 PID 以避免多个 Claude Code 实例冲突
export function getSwarmSocketName(): string {
  return `claude-swarm-${process.pid}`
}
```

&emsp;&emsp;这个设计的意图是**隔离**——Agent Teams 创建的 pane 不会污染你日常使用的 tmux 工作环境。代价是你必须用 `-L` 参数显式指定 socket 才能看到或操作这些 session：

&emsp;&emsp;并发隔离的完整机制是双层的：**socket 层**用 `claude-swarm-${pid}` 隔离不同 Claude Code 实例（每个实例的 PID 不同，socket 名也不同，互不可见）；**session 层**在同一个 socket 内固定用 `claude-swarm` 作为 session 名（源码 `TmuxBackend.ts:471` 用 `hasSessionInSwarm()` 检查 session 是否存在，存在则复用、不存在则用 `new-session` 创建）。所以即使你同时开多个 Claude Code、每个都创建 team，每个实例的 swarm session 互不干扰。

> **【常见误区】**：上面命令里的 `<pid>` 是占位符，不是字面量。你需要先用 `ps aux | grep claude` 或 macOS 上的活动监视器找到 Claude Code 进程的实际 PID（如 78342），然后把命令里的 `<pid>` 换成这个数字，例如 `tmux -L claude-swarm-78342 ls`。如果直接执行 `tmux -L claude-swarm-<pid> ls` 会得到 socket 不存在的错误。

```bash
# 列出外部 swarm session（<pid> 替换为启动 Claude Code 时的进程 ID）
tmux -L claude-swarm-<pid> ls

# 连接进去查看 teammate 的实时输出
tmux -L claude-swarm-<pid> a

# 删除整个 swarm（最彻底的清理）
tmux -L claude-swarm-<pid> kill-server
```

&emsp;&emsp;回顾一下：0.4 节里那条 `tmux kill-session -t claude1` 清理的是你**手动创建**的 tmux session。如果你想清理 Agent Teams 自动创建的外部 swarm session，命令的差异就在这个 `-L` 标志上——`-L claude-swarm-<pid>` 指向独立 socket，普通 `tmux` 命令默认走的是用户级的默认 socket，两者互不可见。

**跨平台支持：macOS / Linux / Windows**

&emsp;&emsp;两种 pane 后端对操作系统的支持差异很大，in-process 模式则跨平台通用：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 0-2 Agent Teams 三种后端的跨平台支持</font></p>
<div class="center">

| 后端 | macOS | Linux | Windows |
|------|-------|-------|---------|
| **tmux** | 原生支持（`brew install tmux`） | 原生支持（`sudo apt install tmux` / `sudo dnf install tmux`） | 需通过 WSL 等 Unix 兼容层运行 |
| **iTerm2** | 原生支持（`brew install --cask iterm2`） | 不支持 | 不支持 |
| **in-process** | 支持 | 支持 | 支持 |

</div>

&emsp;&emsp;iTerm2 是 macOS 专属应用，[官方 FAQ](https://iterm2.com/faq.html) 明确声明 "iTerm2 is for macOS only"。所以 Linux 和 Windows 用户**无法使用 iTerm2 后端**——他们的可视化 pane 选项只有 tmux。in-process 模式不依赖任何外部终端工具，所以在三种系统上都可用，是最稳妥的 fallback。

**Windows 用户运行 tmux 的路径**

&emsp;&emsp;tmux 是 Unix 工具，无法在原生 Windows（cmd / PowerShell）运行——这点在 tmux 官方仓库 [Issue #2575](https://github.com/tmux/tmux/issues/2575) 里有明确的社区共识。Claude Code 源码 `registry.ts:275-279` 对 Windows 用户的官方指引就是走 WSL（Windows Subsystem for Linux）：

```typescript
// src/utils/swarm/backends/registry.ts:275-279
case 'windows':
  return `To use agent swarms, you need tmux which requires WSL (Windows Subsystem for Linux).
Install WSL first, then inside WSL run:
  sudo apt install tmux
Then start a tmux session with: tmux new-session -s claude`
```

&emsp;&emsp;Windows 10（build 19041 及以上）和 Windows 11 用户可以一行命令启用 WSL2。注意两点关键约束：①`wsl --install` 必须在**管理员权限的 PowerShell** 中执行（否则会因权限不足失败）；②如果你之前装过 WSL1，需要额外执行 `wsl --set-default-version 2` 把默认版本切到 WSL2。[Microsoft 官方文档](https://learn.microsoft.com/en-us/windows/wsl/install)给出的标准流程是：

```bash
# 1. 在 Windows 的【管理员】PowerShell 里执行（普通 PowerShell 会因权限不足失败）
wsl --install

# 1b. 如果之前已装过 WSL1，把默认版本切到 WSL2
wsl --set-default-version 2

# 2. 重启 Windows 后进入 WSL（wsl --install 默认安装的是 Ubuntu 发行版）
#    在 WSL 的 Ubuntu shell 内安装 tmux
sudo apt update && sudo apt install tmux

# 3. Claude Code 必须在 WSL 内独立安装（不能用 Windows 端装的版本）
#    在 WSL 内启动 claude 后即可使用 Agent Teams
```

&emsp;&emsp;WSL 还有一个**新手最容易踩的坑**：路径差异。WSL 内的 Linux 文件系统是 `/home/<user>/...`，从 WSL 访问 Windows 盘走 `/mnt/c/...`（C 盘）；反过来 Windows 端访问 WSL 文件系统走 `\\wsl$\Ubuntu\home\<user>\...`。**强烈建议把 Claude Code 项目目录放在 WSL 内的 Linux 文件系统下**（如 `/home/<user>/projects/`），不要放在 `/mnt/c/...`——后者会经过 Windows ↔ Linux 文件系统转换，文件读写性能下降几十倍，`git` / `tmux` / Claude Code 的文件监听也可能不稳定。

&emsp;&emsp;除了 WSL 之外，[MSYS2](https://packages.msys2.org/packages/tmux)（`pacman -S tmux`）和 [Cygwin](https://www.cygwin.com/) 这两个社区维护的 Unix 兼容环境也提供 tmux 包。但 Claude Code 源码 `registry.ts:275-279` 只对 WSL 路径给出了官方指引，MSYS2/Cygwin 路径**未被源码层面覆盖，使用风险自担**——可能遇到 Claude Code CLI 在这两个环境下的路径解析、信号处理、文件锁行为与原生 Linux 不一致的问题。新装环境建议直接走 WSL2。

> **【跨平台选型建议】**：macOS 用户优先 iTerm2 后端（鼠标点击切 pane，无需记 tmux 快捷键）；Linux 用户只能用 tmux，但 tmux 在 Linux 上是原生工具，体验和 macOS 上一致；Windows 用户走 WSL2 + Ubuntu + tmux 的路径，本质上是在 Windows 里跑一个 Linux 子系统。三种系统都支持 in-process 模式——它不依赖任何外部终端工具，只要能跑 Claude Code 就能用 Agent Teams，是跨平台兼容性最好的兜底方案。

> **【学完本节你已经掌握】**：你现在能在 macOS / Linux / Windows 任一平台上判断该选哪个后端（in-process 全平台兜底、tmux 跨平台主力、iTerm2 仅 macOS）；能用 `tmux ls` / `tmux a` / `tmux kill-session` 完整管理 session 生命周期；能识别 Agent Teams 创建的外部 swarm session（独立 socket、需 `-L claude-swarm-<pid>` 才能看到）。这一节是 0.4-0.6 三个 hands-on 的「跨平台与运维补充」，从下一章起我们进入 Agent Teams 的架构分析层。

---

## <center>第一章：Coordinator 编排者模式的天花板</center>

&emsp;&emsp;在进入 Agent Teams 之前，我们需要先回到第三节课第七章留下的那个「集中 vs 分治」的张力上。Coordinator 模式的核心架构是 hub-and-spoke（集中辐射）：一个中心编排者握住全局视野，向各个 worker 点对点派活，worker 执行完返回摘要就销毁。这套模式在大多数场景下运转良好，但它有一个结构性的天花板——当 Agent 之间的协作不再是「派活-收活」的单次交互时，hub-and-spoke 就开始吃力了。

### 1.1 三类场景戳破 hub-and-spoke 的局限

&emsp;&emsp;先打碎一个直觉——「Coordinator 不是已经能管多个 Agent 了吗？为什么还需要新机制？」答案是：**Coordinator 能管，但只能管一次性的活**。以下三类场景会让 hub-and-spoke 架构的局限暴露无遗。

&emsp;&emsp;**场景一：多轮协商**。想象一个 writer 和 reviewer 的协作流程——writer 写完初稿，reviewer 审查后给出修改意见，writer 修改后再提交，reviewer 再审……这种多轮来回需要双方持久化存在，记住之前的交互历史。但在 Coordinator 模式下，每个子 Agent 执行完就销毁，reviewer 给完意见后自己也消失了——下一轮审查只能 `spawn`（启动）一个新的 reviewer，它对之前审了什么一无所知。Coordinator 不得不充当两者之间的中转站，把所有上下文在每一轮都复述一遍，链路越长效率越差。

&emsp;&emsp;**场景二：动态任务认领**。假设你有一个包含 20 个子任务的大型项目，其中一些任务之间有依赖关系（任务 C 必须等任务 A 和 B 完成后才能开始）。在 Coordinator 模式下，所有调度决策都压在 Coordinator 一个角色身上——它需要持续追踪哪个任务完成了、哪个被阻塞了、哪个可以开始了。任务越多，Coordinator 的认知负担越重。如果 worker 能自己看到任务列表、自主认领可做的任务，调度的负担就能从中心角色分散出去。

&emsp;&emsp;**场景三：持久化状态**。在 Coordinator 模式下，worker 是「用完即抛」的——它不记得上一次被调用时做了什么。但有些角色天然需要持久化状态：一个负责代码审查的 Agent 需要记住它这次审查的标准和之前发现的问题模式；一个负责项目管理的 Agent 需要持续追踪进度。每次 spawn 一个新实例，这些上下文都得从零重建。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260526201643986.png" width=50%></div>

### 1.2 两种模式的本质区别

&emsp;&emsp;把 Coordinator 模式和 Agent Teams 模式放在一起比较，你会看到一组非常清晰的结构差异。这里不是「新版好旧版差」的故事——两种模式解决的是不同层次的协作问题。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 1-1 Coordinator 模式 vs Agent Teams 模式核心对比</font></p>
<div class="center">

| 维度 | Coordinator 模式（第三节课已讲） | Agent Teams 模式（本课新增） |
|------|----------------------------------|-------------------------------|
| 通信拓扑 | hub-and-spoke：Coordinator → Worker → return | P2P mesh：任意 teammate 双向通信 |
| Agent 生命周期 | 一次性（spawn → run → die） | 持久化（idle 与 active 反复切换） |
| 任务管理 | 隐式（Coordinator 的 prompt 里分活） | 显式共享 TaskList（TaskCreate / TaskUpdate / TaskList） |
| 协调方式 | 中心化（Coordinator 独占全局视野） | 分布式（任何 teammate 可认领/完成 task） |
| 通信寻址 | 按 agent ID 单向指令 | 按 teammate name 双向消息，支持广播 |

</div>

&emsp;&emsp;用一个类比帮你记住这组区别：**Coordinator 模式像项目经理开会派活**——员工做完交报告就走，做完就走；**Agent Teams 模式像常驻团队在 Slack 群组里协作**——成员一直在线，自己认领任务，遇到问题直接 @ 同事，做完一件事进入 idle 等下一件，而不是消失。

### 1.3 不是替代，是分层

&emsp;&emsp;这里必须提前澄清一个容易踩的坑：Agent Teams **不是** Coordinator 的替代品，而是在 Coordinator **之上**的一层。team lead 这个角色本质上**就是** Coordinator——它保留了全局视野和编排决策能力；teammate 则是在 Coordinator 手下的 worker 基础上，获得了持久化和 P2P 通信的能力。底层的子 Agent 隔离、Fork 缓存共享、极短摘要回传这些第三节课讲过的机制**完全不变**。

> **【常见误区】**：Agent Teams 和 Coordinator 不是互斥关系。一个 team 里可以混用两种模式——需要多轮交互的 Agent（如 writer 和 reviewer）用 teammate 模式，纯执行一次性任务的 Agent（如格式转换器）仍然用传统 dispatch 模式。判断标准很简单：**需要多轮交互或持久化状态的用 teammate，一次性执行完即可的用 `dispatch`（派发）**。

&emsp;&emsp;还有一点需要提前声明：Agent Teams 目前是 `Claude Code` 的**实验性功能**（experimental），受 feature flag 门控，普通用户需要显式启用才能使用（第五章会讲启用机制和张力升级）。这意味着本课所有的源码分析都是「读源码理解设计」，不包含可执行的 Python 演示 cell——你无法在标准环境中复现 Agent Teams 的运行行为，但你完全可以从源码层面理解它的设计思想。

&emsp;&emsp;说「可以混用两种模式」还不够——你需要一个判断框架，否则每次遇到具体场景都得靠感觉拍板。这里给你三条判断规则，直接落到决策层面：

&emsp;&emsp;**规则一：多轮交互 → `teammate`**。如果两个 Agent 需要反复来回协商（比如 `writer` 写完初稿，`reviewer` 给出修改意见，`writer` 再修改，再提交审查……），这种「多轮闭环」就需要双方持久化存在，用 `teammate` 模式。

&emsp;&emsp;**规则二：跨任务持久化状态 → `teammate`**。如果一个 Agent 需要在多个任务之间保留上下文——比如 `quality-guard` 需要记住这次审查的标准，或者 `project-manager` 需要连续追踪跨任务进度——就用 `teammate`。这类角色一旦销毁重建，上下文就断了。

&emsp;&emsp;**规则三：一次性执行完即走 → `dispatch`**。如果一个 Agent 收到指令、执行完、返回结果，整个生命周期就结束了，不需要和任何其他 Agent 直接对话，用传统 `dispatch`（派发）即可。

&emsp;&emsp;一个混合模式的实际场景是这样的：在一个课件生成团队里，`designer`（设计大纲）、`architect`（规划结构）、`builder`（生成内容）这些角色每次都是一次性执行完就走的——它们用 `dispatch`；而 `quality-guard`（持续把关质量标准）和 `writer`（需要多轮打磨内容）这类角色则用 `teammate`，让它们保持在线并可以互相发消息协商。这不是教学归纳出来的经验，而是 `Claude Code` 源码设计意图的直接反映——`TeamCreateTool.ts` 的 `members` 数组支持动态加减成员，说明 `teammate` 的身份在架构上就被设计为可变的持久化角色，而不是一次性的执行单元。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260526201547209.png" width=50%></div>

> **【常见误区】**：你现在能讲清 Coordinator 模式的三类天花板场景（多轮协商 / 动态认领 / 持久化状态），能用一句话区分两种模式的本质（一次性 dispatch vs 持久化 P2P），也知道了 Agent Teams 是 Coordinator 之上的分层而不是替代。带着这个认知动力，我们进入第二章，看 Agent Teams 用了哪些新机制来解决这些问题。

---

## <center>第二章：Agent Teams 核心机制</center>

&emsp;&emsp;第一章建立了「旧模式为什么不够用」的认知，接下来我们进入 Agent Teams 的核心机制层。这一章解决两个问题：一是 Agent Teams 提供了哪些工具（6 个），二是这些工具背后的两个关键机制——SendMessage 的 P2P 寻址和 Mailbox 的异步投递——是怎么工作的。我们先用一张全景表建立概览，再逐一深入最重要的两个机制。

> **【关于源码中的 `swarm` 命名】**：你在后面的源码路径里会反复看到 `swarm` 这个词（如 `src/utils/swarm/`、`agentSwarmsEnabled.ts`）——这是 Claude Code 源码内部对 Agent Teams 机制使用的**工程代号**，和我们课件中说的"Agent Teams"指的是**同一套机制**。环境变量用 `AGENT_TEAMS`，内部源码用 `swarm`。知道这个对应关系后，看到 `swarm` 就不会困惑了。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260526201544403.png" width=50%></div>

### 2.1 六个工具一览

&emsp;&emsp;Agent Teams 的核心能力由 6 个工具承载，每个工具对应 `src/tools/` 下的一个独立目录。需要注意的是，`TaskCreate` / `TaskUpdate` / `TaskList` 这三个任务管理工具的 `isEnabled()` 实际使用 `isTodoV2Enabled()` 门控（属于通用任务系统 TodoV2），而非 `isAgentSwarmsEnabled()`——它们在 Agent Teams 之外也可用；只有 `TeamCreate` / `TeamDelete` 和 `SendMessage` 的 teammate 寻址功能是 Agent Teams 专属的。下面这张表你可以当作速查手册——后续几章会逐一深入最重要的几个。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 2-1 Agent Teams 六个核心工具</font></p>
<div class="center">

| 工具 | 源码路径 | 职能 | 关键参数 |
|------|----------|------|----------|
| TeamCreate | `src/tools/TeamCreateTool/TeamCreateTool.ts` | 创建 team 配置 + 共享 TaskList 目录 | `team_name`, `description` |
| TeamDelete | `src/tools/TeamDeleteTool/TeamDeleteTool.ts` | 清理 team 配置和 task 目录 | 自动识别当前 team |
| SendMessage | `src/tools/SendMessageTool/SendMessageTool.ts`（917 行） | P2P 通信，按 teammate name 寻址 | `to`（name 或 `"*"` 广播）, `message` |
| TaskCreate | `src/tools/TaskCreateTool/TaskCreateTool.ts` | 创建共享任务 | `subject`, `description` |
| TaskUpdate | `src/tools/TaskUpdateTool/TaskUpdateTool.ts` | 认领 / 完成 / 阻塞任务 | `taskId`, `owner`, `status`, `addBlockedBy` |
| TaskList | `src/tools/TaskListTool/TaskListTool.ts` | 查看所有任务状态 | 无参数 |

</div>

&emsp;&emsp;注意 SendMessage 这个工具——它在 `Claude Code` 里一共有 917 行代码，是这 6 个工具中体量最大的。这不是偶然的：通信是多 Agent 系统最复杂的部分，SendMessage 需要处理寻址、投递、广播、结构化协议消息等多种场景，单看行数就能感受到它在整个系统中的分量。

### 2.2 TeamCreate 做了什么

&emsp;&emsp;在进入通信机制之前，我们先快速看一下 `TeamCreate` 在源码层面做了哪两件事——这能帮你建立「一个 team 的物理存在形式」的概念。

&emsp;&emsp;`TeamCreateTool.ts` 的 `call` 方法做两件事。第一，创建 `~/.claude/teams/{team-name}/config.json`，里面存放 team 的核心配置——`leadAgentId`（team lead 的 ID，格式为 `team-lead@teamName`）和 `members` 数组（每个成员的 `name`、`agentId`、`agentType`、`color` 等字段）。第二，创建 `~/.claude/tasks/{team-name}/` 目录，作为共享 TaskList 的物理存储位置。

&emsp;&emsp;下面是 `TeamCreateTool.ts` 第 146 行的关键调用，它定义了 team lead 的 ID 生成规则：

```typescript
// TeamCreateTool.ts:146
// formatAgentId 将常量 TEAM_LEAD_NAME 与 team 名称拼接为唯一标识
// 格式：team-lead@teamName
const leadAgentId = formatAgentId(TEAM_LEAD_NAME, finalTeamName)
```

&emsp;&emsp;这行代码揭示了一个重要的设计决策：team lead 的 ID **不是随机生成的**，而是由固定的 `TEAM_LEAD_NAME` 常量（值为 `"team-lead"`）和 team 名称组合而成。这意味着一个 team 里有且只有一个 team lead——这个角色的唯一性是由 ID 生成规则在结构上保证的。

### 2.3 SendMessage 的 P2P 寻址——与第三节课的关键差异

&emsp;&emsp;先打碎一个直觉——「SendMessage 不是第三节课第七章就讲过了吗？」答案是：**同一个工具名，寻址方式完全不同**。

&emsp;&emsp;第三节课讲的 SendMessage 是 **Coordinator → Worker 单向指令**。你回忆一下第三节课第 7.2 节的内容：Coordinator 用 `<task-id>` 作为 `to` 参数，给特定 worker 发指令。worker 执行完返回结果就销毁了——这是一条单向的、一次性的通信链路。

&emsp;&emsp;Agent Teams 里的 SendMessage 变成了 **任意 teammate 到任意 teammate 的双向消息**。差异集中在三个点：

&emsp;&emsp;**第一，`to` 字段的寻址方式变了**。不再用 `task-id`，而是用 **teammate name**——就是 `config.json` 里 `members` 数组中每个成员的 `name` 字段。这意味着任何 teammate 都可以直接按名字找到任何其他 teammate，不再需要经过 Coordinator 中转。

&emsp;&emsp;**第二，支持广播**。`to` 参数可以填 `"*"`，一条消息同时发给所有 teammate。这在 Coordinator 模式下是不存在的——Coordinator 模式的通信永远是一对一。

&emsp;&emsp;下面是 `SendMessageTool.ts` 第 69-75 行对 `to` 参数的描述。源码实际使用 `feature('UDS_INBOX')` 条件分支——当 UDS 功能开启时描述更长（包含 Unix Domain Socket 和 Remote Control 两种额外寻址方式），未开启时是下面这个精简版。我们展示的是非 UDS 分支（第 74 行），因为它清晰定义了 Agent Teams 的核心寻址规则：

```typescript
// SendMessageTool.ts:69-75（简化展示，省略 UDS 分支）
// to 参数描述：接收者可以是 teammate name 或 "*" 表示广播
// 这与 Coordinator 模式下用 task-id 寻址是完全不同的机制
.describe(
  'Recipient: teammate name, or "*" for broadcast to all teammates'
)
```

&emsp;&emsp;**第三，支持结构化协议消息**。除了普通文本消息，Agent Teams 的 SendMessage 支持三种结构化协议消息（`SendMessageTool.ts:46-65`）：`shutdown_request`（关闭请求）、`shutdown_response`（关闭响应）、`plan_approval_response`（计划审批响应）。这些协议消息让 teammate 之间可以进行有序的协商流程——比如 team lead 发 `shutdown_request` 让某个 teammate 优雅退出，teammate 回 `shutdown_response` 确认或拒绝。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 2-2 SendMessage 寻址方式对比</font></p>
<div class="center">

| 维度 | Coordinator 模式（第三节课） | Agent Teams 模式（本课） |
|------|------------------------------|--------------------------|
| `to` 参数值 | `<task-id>`（worker 的 agent ID） | teammate name 或 `"*"` |
| 通信方向 | Coordinator → Worker（单向指令） | 任意 teammate ↔ 任意 teammate（双向） |
| 广播能力 | 无 | `to: "*"` 广播给所有 teammate |
| 协议消息 | 无 | `shutdown_request` / `shutdown_response` / `plan_approval_response` |
| 生命周期 | Worker 执行完即销毁 | Teammate 持久化，idle 后可被再次唤醒 |

</div>

&emsp;&emsp;这张表清晰地呈现了从「指令式调度」到「协商式协作」的跃迁。寻址方式的变化不是表面的参数改名，而是底层协作范式的根本转变——从「中心向外辐射」变成了「任意节点互连」。

### 2.4 Mailbox 异步投递设计

&emsp;&emsp;teammate 之间发消息不是直接的函数调用（RPC），而是通过一个**文件系统邮箱**异步投递——这个设计决策本身就值得你深入理解。

&emsp;&emsp;整个 Mailbox 机制集中在一个文件里：`src/utils/teammateMailbox.ts`，实测 1183 行，是 Agent Teams 通信子系统中体量最大的文件（整个 `swarm/` 目录下 `inProcessRunner.ts` 有 1552 行更长，但那是 in-process 后端的执行引擎，第四章会讲到）。它为什么这么大？因为文件系统邮箱需要处理大量的并发安全问题——多个 teammate 可能同时写入同一个收件箱、读取和写入可能竞争、idle 通知可能和正常消息交错——这些边界 case 占了大量代码。

&emsp;&emsp;核心流程由三个函数串起来：

```typescript
// src/utils/teammateMailbox.ts
// 收件箱物理路径：~/.claude/teams/{team_name}/inboxes/{agent_name}.json
// 注意：mailbox 文件在 teams/{team}/inboxes/ 下，与 tasks/{team}/ 下的 TaskList 是两个独立目录

// L134: 写入消息——把消息写入接收者的收件箱文件
// 每个 teammate 有独立收件箱路径：getInboxPath(agentName, teamName)
export async function writeToMailbox(/* ... */)

// L115: 读取未读消息——拉取当前 teammate 收件箱中的未读消息
// 返回值区分已读和未读，只返回新增的
export async function readUnreadMessages(/* ... */)

// L410: 创建 idle 通知——teammate 停止时自动通知 team lead
// 由 teammateInit.ts 的 Stop hook 自动触发，不需要手动调用
export function createIdleNotification(/* ... */)
```

&emsp;&emsp;这三个函数构成了一条完整的投递链路：teammate A 调用 `writeToMailbox()` 把消息写入 teammate B 的收件箱（具体路径是 `~/.claude/teams/{team}/inboxes/{agent}.json`，可以在本地直接 `ls` 看到）→ teammate B 调用 `readUnreadMessages()` 拉取未读消息 → 当 teammate B 完成当前任务停止时，`createIdleNotification()` 自动生成一条 idle 通知，告诉 team lead「我闲下来了，可以接新活」。注意 mailbox 和 TaskList 是物理分离的——mailbox 在 `teams/{team}/inboxes/` 下、TaskList 在 `tasks/{team}/` 下，前者负责点对点消息，后者负责共享任务状态，两者不互通。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260526201547168.png" width=50%></div>

> **【常见误区】**：Mailbox 不是实时通信——它是异步投递。teammate A 发出消息后不会阻塞等待 teammate B 的回复，而是继续做自己的事。teammate B 在下一次轮询收件箱时才会看到新消息。idle 通知也是自动的——当 teammate 的 Stop hook 触发时（`teammateInit.ts:28` 的 `initializeTeammateHooks` 注册了这个 hook），`createIdleNotification()` 会自动调用，teammate 不需要主动「报告」自己闲了。

&emsp;&emsp;为什么选择文件系统而不是内存队列或数据库？这是一个刻意的工程取舍：文件系统邮箱的最大优势是**零外部依赖**——不需要 Redis、不需要消息中间件、不需要数据库。`Claude Code` 作为一个 CLI 工具，它的运行环境可能是开发者的个人笔记本，不能假设有任何额外的基础设施。文件系统是一种在几乎任何环境下都可用的持久化介质，不需要额外安装数据库或消息队列。代价是性能和并发安全处理的复杂度更高——这也解释了为什么 `teammateMailbox.ts` 有 1183 行那么多代码。

> **【学完本章你已经掌握】**：你现在能画出 Agent Teams 的 6 个工具各自的位置，能精确区分 SendMessage 在 Coordinator 模式和 Agent Teams 模式下的寻址差异（`task-id` vs `teammate name`），也知道了 Mailbox 为什么选择文件系统异步投递而不是直连 RPC。下一章我们进入协调层——共享 TaskList 是 Agent Teams 与 Coordinator 模式在「任务管理」维度上最大的结构差异。

---

## <center>第三章：共享 TaskList 与分布式协调</center>

&emsp;&emsp;第二章打开了通信层——teammate 之间怎么发消息、消息怎么投递。但光能发消息还不够，多个 Agent 协作时还需要回答一个更核心的问题：**谁来决定做什么、谁在做什么、什么时候做完了？** 在 Coordinator 模式下，这些问题的答案全在 Coordinator 的 prompt 里——它用自然语言分配任务，用自己的上下文追踪进度，这是「隐式任务管理」。Agent Teams 引入了一套完全不同的方案：**显式共享 TaskList**——所有任务以结构化数据的形式存储在文件系统里，任何 teammate 都可以查看和操作。

### 3.1 Task 数据模型

&emsp;&emsp;先看 Task 的数据结构。`src/utils/tasks.ts` 第 76 行定义了完整的 `TaskSchema`，每个 Task 包含以下字段：

```typescript
// src/utils/tasks.ts:76-88
// TaskSchema 定义了一个 Task 的完整数据结构
// 存储位置：~/.claude/tasks/{team-name}/ 目录下的 JSON 文件
export const TaskSchema = lazySchema(() =>
  z.object({
    id: z.string(),                    // 自增 ID，字符串形式
    subject: z.string(),               // 任务标题（一句话描述）
    description: z.string(),           // 详细描述（包含执行要求）
    activeForm: z.string().optional(), // 进行时描述，用于 UI spinner（如 "Running tests"）
    owner: z.string().optional(),      // agent ID
    status: TaskStatusSchema(),        // 'pending' | 'in_progress' | 'completed'
    blocks: z.array(z.string()),       // 本 task 阻塞的其他 task ID 列表
    blockedBy: z.array(z.string()),    // 阻塞本 task 的其他 task ID 列表
    metadata: z.record(z.string(), z.unknown()).optional(), // 自定义元数据
  }),
)
```

&emsp;&emsp;这里有两个字段特别值得你注意。

&emsp;&emsp;**第一，`status` 字段只有 3 个正式枚举值**：`pending`（等待中）、`in_progress`（进行中）、`completed`（已完成）。这在 `tasks.ts` 第 69 行有明确定义：

```typescript
// src/utils/tasks.ts:69
// 注意：只有 3 个值，不含 'deleted'
// 'deleted' 是 TaskUpdateTool 的额外扩展值，不在正式枚举内
export const TASK_STATUSES = ['pending', 'in_progress', 'completed'] as const
```

> **【常见误区】**：你可能在 `TaskUpdateTool.ts` 中看到 `deleted` 状态——那是 TaskUpdate 工具为了支持「删除任务」而追加的扩展值，不在 `TaskStatusSchema` 的正式枚举里。区分很重要：`TaskStatusSchema` 定义的是 Task 的**合法存储状态**（3 个值），`TaskUpdateTool` 的输入 schema 扩展了 `deleted` 作为一个**操作动作**。

&emsp;&emsp;**第二，`owner` 字段的源码注释是 `// agent ID`**，但实际自动填入的是 agent 的 **name**（不是完整的 `agentId`）。`TaskUpdateTool.ts` 第 194 行的代码证实了这一点：当 teammate 标记任务为 `in_progress` 但没有显式指定 `owner` 时，系统会自动调用 `getAgentName()` 填入当前 teammate 的 name——你看到 `owner: "researcher"` 而不是 `owner: "researcher@my-team"`。这里源码注释与实际行为有微妙差异，是一个值得注意的工程细节。

### 3.2 blockedBy 依赖链——声明式依赖管理

&emsp;&emsp;`blocks` 和 `blockedBy` 这对字段是 TaskList 最精巧的设计。它们实现了一种**声明式依赖管理**：你不需要写代码来控制任务执行顺序，只需要在创建任务时声明「任务 C 被任务 A 和 B 阻塞」，系统就会自动保证 C 只有在 A 和 B 都完成后才变为可认领状态。

&emsp;&emsp;来看一个具体的依赖关系示例。假设一个课件生成 team 有以下 4 个任务：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 3-1 TaskList 依赖关系示例</font></p>
<div class="center">

| Task ID | subject | status | owner | blockedBy | blocks |
|---------|---------|--------|-------|-----------|--------|
| 1 | 编写教学计划 | completed | planner | [] | [3] |
| 2 | 收集源码材料 | in_progress | researcher | [] | [3] |
| 3 | 生成课件初稿 | pending | — | [1, 2] | [4] |
| 4 | 审查课件质量 | pending | — | [3] | [] |

</div>

&emsp;&emsp;Task 3「生成课件初稿」的 `blockedBy` 是 `[1, 2]`——它必须等 Task 1 和 Task 2 都完成后才可以被认领。当 Task 2 的 status 变为 `completed` 时，Task 3 的 `blockedBy` 列表中所有前置任务都已完成，它就变成了可认领状态。这种声明式依赖比 Coordinator 模式下「在 prompt 里用自然语言描述执行顺序」精确得多——依赖关系是结构化数据，不是靠 LLM 理解自然语言来保证的。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260526201551616.png" width=50%></div>

### 3.3 协调流程：从创建到完成

&emsp;&emsp;把 Task 数据模型和依赖链组合起来，完整的协调流程是这样的：

&emsp;&emsp;**步骤一**：Team lead 用 `TaskCreate` 创建任务，设定 `subject`、`description`，必要时通过 `addBlockedBy` 声明依赖关系。

&emsp;&emsp;**步骤二**：Teammate 用 `TaskList` 查看当前所有任务的状态。一个任务「可认领」的条件是：`status` 为 `pending`、没有 `owner`、`blockedBy` 列表为空或列表中所有 task 都已 `completed`。

&emsp;&emsp;**步骤三**：Teammate 用 `TaskUpdate(owner=自己的name)` 认领任务，同时将 `status` 改为 `in_progress`。如果 teammate 只设了 `status` 为 `in_progress` 但没有显式指定 `owner`，系统会自动把当前 teammate 的 name 填入 `owner` 字段。

&emsp;&emsp;**步骤四**：完成后 `TaskUpdate(status='completed')` 标记完成。被本 task 阻塞的下游任务（`blocks` 字段列出的 task）自动解锁。

&emsp;&emsp;这里有一个设计细节值得你关注：**任务按 ID 顺序优先认领**（低 ID 先做）。为什么？因为先创建的任务往往是后续任务的前置条件——按 ID 顺序认领相当于默认按依赖顺序执行，减少了不必要的阻塞等待。

&emsp;&emsp;下面这张时序图把上述四个步骤串成一个完整场景——Team Lead 创建 3 个有依赖关系的 task，两个 Teammate 并行认领和执行，Task 3 在前置任务全部完成后自动解锁：

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260526201549721.png" width=50%></div>

### 3.4 隐式 vs 显式——与 Coordinator 模式的结构对比

&emsp;&emsp;到这里我们可以做一个「隐式 vs 显式」的结构对比了。这不是「哪个更好」的问题，而是两种截然不同的协调哲学。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 3-2 隐式任务管理 vs 显式 TaskList</font></p>
<div class="center">

| 维度 | Coordinator 模式（隐式） | Agent Teams TaskList（显式） |
|------|--------------------------|------------------------------|
| 任务分配 | Coordinator 在 prompt 中用自然语言分活 | Team lead 用 TaskCreate 创建结构化任务 |
| 依赖管理 | Coordinator 靠自己的上下文记住执行顺序 | `blockedBy` 声明式依赖，系统自动保证 |
| 状态可见性 | 只有 Coordinator 知道全局进度 | 任何 teammate 调用 TaskList 都能看到全部 |
| 动态调整 | Coordinator 重新 dispatch（需要重新理解上下文） | 任何 teammate 可创建新 task 或修改依赖 |
| 认领方式 | Coordinator 指定谁做什么 | Teammate 自主查看可用任务并认领 |

</div>

&emsp;&emsp;显式 TaskList 的核心价值是**状态的可观测性**。在 Coordinator 模式下，只有 Coordinator 一个角色知道「现在整体进度到哪了」，其他 worker 只知道自己那份活。在 Agent Teams 模式下，任何 teammate 调用 `TaskList` 都能看到所有任务的状态、依赖关系和认领情况——这使得「分布式决策」成为可能。一个 teammate 看到自己前置任务都完成了，就可以自主认领下一个任务，不需要等 team lead 来通知。

> **【学完本章你已经掌握】**：你现在能画出 Task 从 `pending` → `in_progress` → `completed` 的完整状态机，能解释 `blockedBy` 声明式依赖链的工作原理，也能精确区分 Coordinator 的隐式任务管理和 Agent Teams 的显式 TaskList 在可观测性上的结构差异。下一章我们向下走一层——Agent Teams 的执行基础设施：三种后端和身份系统。

---

## <center>第四章：三层后端与身份系统</center>

&emsp;&emsp;前两章我们看了 Agent Teams 的「上层建筑」——怎么通信（SendMessage + Mailbox）、怎么协调（TaskList + blockedBy）。这一章我们看它的「地基」：teammate 到底运行在什么环境里？系统怎么知道「我是谁」？一个 teammate 完成任务后怎么进入待命状态？这三个问题分别对应三个源码模块：后端类型（backends）、身份系统（teammate.ts + teammateContext.ts）、生命周期管理（teammateInit.ts）。

### 4.1 三种后端类型

&emsp;&emsp;Agent Teams 支持三种运行后端，定义在 `src/utils/swarm/backends/types.ts` 第 9 行：

```typescript
// src/utils/swarm/backends/types.ts:9
// 三种后端类型：决定 teammate 运行在什么环境
export type BackendType = 'tmux' | 'iterm2' | 'in-process'
```

&emsp;&emsp;这三种后端的差异本质上是**隔离粒度 vs 性能开销**的取舍：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 4-1 三种后端类型对比</font></p>
<div class="center">

| 后端 | 隔离方式 | 适用场景 | 优势 | 代价 |
|------|----------|----------|------|------|
| tmux | 每个 teammate 独立进程，在 tmux pane 中运行 | 可视化调试，需要看到每个 teammate 的实时输出 | 天然进程隔离，崩溃不影响其他 teammate | 需要 tmux 环境，进程启动开销较大 |
| iTerm2 | 每个 teammate 独立进程，在 iTerm2 tab 中运行 | macOS 开发者偏好 iTerm2 时使用 | 与 tmux 类似，UI 体验更友好 | 仅限 macOS + iTerm2 |
| in-process | 所有 teammate 在同一个 Node.js 进程中运行 | 性能优先，不需要可视化 | 无进程启动开销，通信更快 | 需要 `AsyncLocalStorage` 做上下文隔离 |

</div>

&emsp;&emsp;`tmux` 和 `iTerm2` 本质上是同一种方案的两个变体——都是「一个 teammate 一个进程」，区别只在终端模拟器的选择。真正值得你理解的是 `in-process` 后端，因为它面对一个独特的工程挑战：**多个 teammate 跑在同一个 Node.js 进程里，怎么保证它们的上下文不互相污染？**

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260526201544425.png" width=50%></div>

### 4.2 in-process 隔离：AsyncLocalStorage 的精妙

&emsp;&emsp;答案在 `src/utils/teammateContext.ts` 里——它用 Node.js 内置的 `AsyncLocalStorage` 来隔离同一进程内多个 teammate 的上下文。

```typescript
// src/utils/teammateContext.ts:16-17
// AsyncLocalStorage 是 Node.js async_hooks 模块提供的存储机制
// 每个异步调用链自动继承自己的 store，不与其他链共享
import { AsyncLocalStorage } from 'async_hooks'

// :41
// 创建全局的 AsyncLocalStorage 实例
// 每个 in-process teammate 在自己的 run() 上下文里有独立的 store
const teammateContextStorage = new AsyncLocalStorage<TeammateContext>()
```

&emsp;&emsp;`AsyncLocalStorage` 的工作原理你可以这样理解：当代码执行 `teammateContextStorage.run(context, fn)` 时（第 63 行），`fn` 内部所有的同步和异步操作都能通过 `getStore()` 拿到传入的 `context`——而且不同 `run()` 调用之间的 store 是完全隔离的。这意味着即使两个 teammate 在同一个 Node.js 事件循环里交替执行，它们各自的身份信息（「我是谁」「我属于哪个 team」）永远不会搞混。

> **【常见误区】**：in-process 后端是「共进程不共上下文」——它们共享同一个 Node.js 进程（共享内存、共享事件循环），但通过 `AsyncLocalStorage` 实现了逻辑上的上下文隔离。注意这不是完美的进程级隔离：如果一个 teammate 抛出未捕获异常导致进程崩溃，所有 in-process teammate 都会一起倒下。这就是为什么 tmux/iTerm2 后端在需要高可靠性的场景下仍然有价值。

### 4.3 身份系统——「我是谁」的解析链路

&emsp;&emsp;每个 teammate 需要知道自己是谁、属于哪个 team——这些身份信息存储在 `src/utils/teammate.ts` 第 44 行定义的 `dynamicTeamContext` 结构体里：

```typescript
// src/utils/teammate.ts:44-51
// dynamicTeamContext 存储 teammate 的完整身份信息
// 6 个字段覆盖了身份标识、归属、UI 和权限控制
let dynamicTeamContext: {
  agentId: string          // 唯一标识，格式 agentName@teamName
  agentName: string        // 显示名，SendMessage 的 to 参数用这个
  teamName: string         // 所属 team 名称
  color?: string           // UI 颜色标识（可选）
  planModeRequired: boolean // 是否需要 plan mode 审批
  parentSessionId?: string // 父 session ID（可选）
} | null = null
```

&emsp;&emsp;这 6 个字段中，最关键的是 `agentName` 和 `teamName`——前者是 SendMessage 寻址的基础（`to` 参数就是 `agentName`），后者决定了这个 teammate 的 Mailbox 收件箱和 TaskList 的存储路径。`agentId` 是两者的组合（格式 `agentName@teamName`），提供全局唯一标识。

&emsp;&emsp;身份解析有一条明确的优先级链，定义在 `teammate.ts` 第 86 行的注释中：

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260526201547188.png" width=50%></div>

&emsp;&emsp;**优先级 1：`AsyncLocalStorage`**（in-process teammate）。如果当前代码运行在 `teammateContextStorage.run()` 的上下文里，直接从 store 中取身份信息。这是 in-process 后端的专属通道。

&emsp;&emsp;**优先级 2：`dynamicTeamContext`**（tmux/iTerm2 teammate）。这些后端通过 CLI 参数（`--agent-id`、`--team-name` 等）把身份信息传给子进程，启动时写入 `dynamicTeamContext` 模块变量。

&emsp;&emsp;如果两个来源都没有值，说明当前 session 不是作为 teammate 运行的。

### 4.4 三个关键守卫函数

&emsp;&emsp;基于身份信息，`teammate.ts` 提供了三个守卫函数供系统各处调用：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 4-2 身份守卫函数</font></p>
<div class="center">

| 函数 | 位置 | 返回值 | 用途 |
|------|------|--------|------|
| `isTeammate()` | `teammate.ts:125` | `boolean` | 判断当前 session 是否作为 teammate 运行 |
| `isTeamLead()` | `teammate.ts:171` | `boolean` | 判断当前 session 是否是 team lead |
| `getAgentId()` | `teammate.ts:88` | `string` 或 `undefined` | 获取当前 teammate 的完整 agent ID |

</div>

&emsp;&emsp;其中 `isTeammate()` 是使用最广泛的——系统里大量的条件分支（是否启用 Mailbox 轮询、是否注册 Stop hook、是否在 UI 中显示 team 信息）都依赖这个函数的返回值。`isTeamLead()` 的判断逻辑稍微复杂一些：主路径是通过 `getAgentId()` 拿到当前 teammate 的完整 ID，然后与 `teamContext.leadAgentId` 做**精确等值比较**（`myAgentId === teamContext.leadAgentId`）；还有一条向后兼容分支——当 `myAgentId` 为 `undefined` 时也返回 `true`（处理最早创建 team 的 session 没有 `CLAUDE_CODE_AGENT_ID` 环境变量的情况）。

### 4.5 生命周期——idle 不是结束

&emsp;&emsp;最后一块拼图是 teammate 的生命周期。`src/utils/swarm/teammateInit.ts` 第 28 行的 `initializeTeammateHooks` 函数在 teammate 启动时注册了一个关键的 Stop hook：

```typescript
// src/utils/swarm/teammateInit.ts:28
// initializeTeammateHooks 在 teammate 启动时被调用
// 它注册一个 Stop hook：当 teammate 停止时自动发送 idle 通知给 team lead
export function initializeTeammateHooks(/* ... */)

// :109
// Stop hook 回调内调用 createIdleNotification
// 告诉 team lead "我闲下来了，可以接新活"
const notification = createIdleNotification(agentName, {/* ... */})
```

&emsp;&emsp;这段代码揭示了 Agent Teams 最核心的生命周期设计：**teammate 的 Stop 不是死亡，而是进入 idle 状态**。传统 Coordinator 模式下，worker 执行完就销毁了——它的生命周期是「spawn → run → die」，是一次性的。Agent Teams 模式下，teammate 的生命周期是「spawn → run → idle → 被新消息唤醒 → run → idle → ...」的循环，是持久化的。

> **【常见误区】**：看到 teammate 进入 idle 不要以为它出了问题。idle 是 Agent Teams 的正常状态——它意味着「当前没有待处理的消息或任务，我在等待新的工作」。team lead 收到 idle 通知后可以决定是给它派新活还是让它继续等待。当 team lead 发送 `shutdown_request` 协议消息时，teammate 才会真正终止。

&emsp;&emsp;team 发现机制也值得一提：`src/utils/teamDiscovery.ts` 提供了查询 team 和 teammate 状态的能力。第 50 行有一个有趣的设计细节——在遍历 team 成员列表时，代码会**跳过 `name === 'team-lead'` 的成员**：

```typescript
// src/utils/teamDiscovery.ts:50
// getTeammateStatuses() 遍历成员时跳过 team lead
// 因为 team lead 不需要被"发现"——它本身就是发现者
if (member.name === 'team-lead') {
  continue
}
```

&emsp;&emsp;为什么要跳过 team lead？因为 `getTeammateStatuses()` 的调用者通常就是 team lead 自己——它需要知道手下各个 teammate 的状态（谁在工作、谁在 idle、谁离线了），但它不需要「发现」自己。这个看似微小的 `continue` 体现了一个设计原则：**team lead 是发现者，不是被发现者**。

> **【学完本章你已经掌握】**：你现在能解释三种后端类型的差异和选择依据，能画出身份解析的优先级链路（`AsyncLocalStorage` > `dynamicTeamContext`），也理解了 idle 不是终止而是持久化等待。Agent Teams 的机制全貌到这里就拼齐了——通信层（SendMessage + Mailbox）、协调层（TaskList + blockedBy）、基础设施层（后端 + 身份 + 生命周期）。最后一章，我们把这些机制放回第三节课的四对张力框架里做认知升级。

---

## <center>第五章：张力升级与展望</center>

&emsp;&emsp;前四章我们从 Coordinator 的天花板出发，逐层拆开了 Agent Teams 的通信、协调和基础设施三层机制。这一章我们做两件事：第一，把四对张力框架映射到 Agent Teams，看看每对张力在新机制下发生了什么变化；第二，看 Agent Teams 的启用机制——feature flag 三层门控的设计意图，以及它为后续课程（第四节课 fufan-cc-flow）留下的桥接口。

### 5.1 四对张力在 Agent Teams 中的映射

&emsp;&emsp;第三节课第九章给了你一把由四对张力构成的尺——能力 vs 约束、信任 vs 约束、静态 vs 动态、集中 vs 分治。这把尺是用来量任何 Agent 系统的，现在我们把它对准 Agent Teams，看看每对张力怎么演化。

> **【关于「四对张力」这个说法】**：这是本系列课程从第三节课第九章开始使用的教学归纳框架，不是 `Claude Code` 源码里的官方术语。但它不是凭空总结的——每对张力都能在源码设计决策中找到直接对应。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>表 5-1 四对张力在 Agent Teams 中的演化</font></p>
<div class="center">

| 张力 | 第三节课的体现 | Agent Teams 中的新体现 |
|------|----------------|------------------------|
| **能力 vs 约束** | 扩展三件套是油门，安全管线是刹车 | teammate 获得 P2P 通信能力（油门加大），同时需要 TaskList 依赖管理和 shutdown 协议来约束（刹车也加厚） |
| **信任 vs 约束** | 越自主越需要可验证关卡 | teammate 可以自主认领任务（信任增加），但 team lead 通过 TaskList 保持全局可见性（约束不松） |
| **静态 vs 动态** | 压缩动态丢，记忆静态留 | `config.json` 是静态注册（team 成员在创建时确定），但 teammate 的状态（idle / active / 离线）是动态变化的——`teamDiscovery.ts` 实时查询 |
| **集中 vs 分治** | Coordinator 集中决策 + 子 Agent 分治执行 | team lead 仍是「中心」（保留全局视野），但 TaskList 允许分治式任务协调——集中决策 + 分治执行的张力**更加明显** |

</div>

&emsp;&emsp;其中最值得你深想的是「集中 vs 分治」这对张力的升级。在第三节课的 Coordinator 模式下，这对张力的表现是「集中决策 + 分治执行」——Coordinator 独自做所有编排决策，worker 只管执行。到了 Agent Teams，分治的范围扩大了——teammate 不仅在执行层分治，在**认领和协调层也有了分治能力**（自主查看 TaskList、自主认领任务、自主发消息给其他 teammate）。但 team lead 仍然保持着全局视野——它能看到所有 task 的状态、收到所有 teammate 的 idle 通知、握有 shutdown 的决定权。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260526201553814.png" width=50%></div>

### 5.2 「有 leader 的 P2P」——核心设计取舍

&emsp;&emsp;先打碎一个直觉——「P2P 是不是意味着无中心？」答案是：**Agent Teams 的 P2P 是有 leader 的 P2P，不是无中心的纯对等网络**。

&emsp;&emsp;team lead 保留了三个关键的「中心」职能：一，**全局视野**——它能看到所有 task 的状态和所有 teammate 的 idle/active 状态；二，**shutdown 权**——只有 team lead 能发送 `shutdown_request` 协议消息要求 teammate 终止；三，**team 生命周期管理**——TeamCreate 和 TeamDelete 都是由 team lead 角色发起的。

&emsp;&emsp;teammate 获得的是**通信层和协调层的 P2P 能力**：可以不经过 team lead 直接给其他 teammate 发消息，可以自主查看 TaskList 并认领任务。这是一个精心设计的折中——**集中决策不丢（team lead 仍有最终控制权），但分治协作的空间大幅扩展**。

&emsp;&emsp;这个设计取舍对你理解（甚至设计）多 Agent 系统有直接的参考价值：纯集中式的 Coordinator 在复杂协作中会成为瓶颈（第一章的三类场景），纯对等的无中心网络又缺乏协调能力（没有人能做全局决策）。「有 leader 的 P2P」是这两种极端之间一个实用的工程取舍点。

&emsp;&emsp;每一种架构取舍都有代价，这节课的风格是双面都讲。P2P 通信带来了效率和灵活性，但它也有三个经过源码核实的结构性局限，值得你在评估多 Agent 系统时认真对待。

&emsp;&emsp;**局限一：没有投递确认（ack）机制**。`writeToMailbox()` 的返回值是 `Promise<void>`——消息写入收件箱文件后函数就完成了。文件锁的 `retry` 机制只保护并发写入时的安全性，它解决的是「两个 Agent 同时写同一个文件」的冲突，而不是「对方是否真的收到了消息」的问题。换句话说，发出去的消息没有超时机制、没有死锁检测、没有任何回执——如果接收方因为某种原因没有轮询收件箱，发送方无从得知。

&emsp;&emsp;**局限二：P2P 直连使 orchestrator 失去修复进度的可见性**。如果 `writer` 和 `reviewer` 之间通过 P2P 消息直接进行多轮修复循环，`team lead` 对这个过程是不可见的——它不知道双方已经来回了几轮、现在到了哪个阶段、是否陷入了死循环。对 `team lead` 而言，这段时间里只是收到了两个 `teammate` 持续处于 `active` 状态的信号，仅此而已。

&emsp;&emsp;**局限三：没有 watchdog 机制**。如果一个 `teammate` 进入 `idle` 状态后始终没有被唤醒——比如它的收件箱文件意外被删除，或者发来的消息路径拼接出错——没有任何自动检测机制会发现这个异常。系统不会产生告警，`team lead` 也不会收到任何通知，这个 `teammate` 就静默地"消失"了。

> **【关于以下工程方向的说明】**：这是基于当前 `Claude Code` 已有机制（`SendMessage` + `TaskUpdate`）自然延伸出的工程方向，**不是 `Claude Code` 当前源码中已实现的模式**，仅供架构评估参考。

&emsp;&emsp;一个自然的改进方向是 **Observable P2P**：保留 `teammate` 之间 P2P 通信的效率，但所有 P2P 消息同时触发一次 `TaskUpdate`，将修复轮次和中间状态写入共享 `TaskList`。这样 `orchestrator` 就可以通过轮询 `TaskList` 保持对整个修复进度的全局可见性，而不是只能靠 `idle` 通知来猜测进展。这不是重新引入中心化，而是在去中心化通信的基础上叠加一层可观测性——效率和可见性不必二选一。

&emsp;&emsp;回到「有 leader 的 P2P」这个设计取舍：以上三个局限恰好证明了保留 `team lead` 全局视野的必要性。如果完全去中心化、让所有 `teammate` 完全对等地通信，这些局限会被进一步放大——没有人能感知到投递失败、没有人能发现某个 `teammate` 静默消失、没有人能把握修复循环的全局进度。**保留 `team lead` 的中心角色，正是对这些结构性局限的一道防护**。

### 5.3 Feature Flag 三层门控

&emsp;&emsp;Agent Teams 不是默认开启的——`src/utils/agentSwarmsEnabled.ts` 第 24 行的 `isAgentSwarmsEnabled()` 函数实现了一套三层门控逻辑：

```typescript
// src/utils/agentSwarmsEnabled.ts:24-44
// isAgentSwarmsEnabled 三层门控：
// 第一层：Anthropic 内部用户直接放行
// 第二层：外部用户需要 env var 或 CLI flag 显式启用
// 第三层：GrowthBook killswitch 作为最终安全阀
export function isAgentSwarmsEnabled(): boolean {
  // 第一层：Ant 内部用户直接返回 true
  if (process.env.USER_TYPE === 'ant') {
    return true
  }

  // 第二层：外部用户需要 env var 或 --agent-teams CLI flag
  if (
    !isEnvTruthy(process.env.CLAUDE_CODE_EXPERIMENTAL_AGENT_TEAMS) &&
    !isAgentTeamsFlagSet()  // 检查 --agent-teams 命令行参数
  ) {
    return false
  }

  // 第三层：GrowthBook killswitch（tengu_amber_flint）
  // 即使外部用户手动启用了，如果 killswitch 关闭也会被拦截
  if (!getFeatureValue_CACHED_MAY_BE_STALE('tengu_amber_flint', true)) {
    return false
  }

  return true
}
```

&emsp;&emsp;三层门控各自的设计意图非常清晰：

&emsp;&emsp;**第一层（内部直通）**：`process.env.USER_TYPE === 'ant'` 检查当前用户是否是 Anthropic 内部员工。内部用户直接返回 `true`，跳过后续所有检查——因为内部用户是这个功能的首批测试者和迭代者。

&emsp;&emsp;**第二层（显式启用）**：源码设计了两种启用入口——环境变量 `CLAUDE_CODE_EXPERIMENTAL_AGENT_TEAMS=1`（第 32 行）和 `--agent-teams` CLI flag（第 11 行的 `isAgentTeamsFlagSet()` 检查 `process.argv`）。但实际上 `--agent-teams` flag 只在 `ant` 内部构建版本的 CLI 参数解析器中注册，外部公开版本会报 `unknown option` 错误——**外部用户只能用环境变量方式**（通过 `~/.claude/settings.json` 的 `env` 字段设置，见第 0 章）。

&emsp;&emsp;**第三层（安全阀）**：GrowthBook 的 `tengu_amber_flint` killswitch（第 39 行）。即使外部用户手动启用了功能，Anthropic 仍然可以通过远程配置关闭它——这是一个「即使用户开了开关，我们也能从远端拉闸」的安全机制。`_CACHED_MAY_BE_STALE` 后缀告诉你这个值是缓存的，可能不是实时最新的，但对于 killswitch 来说这种延迟是可接受的。

> **【常见误区】**：设了环境变量但 `TeamCreate` 工具仍然不出现？不一定是你配置错了——第三层 GrowthBook killswitch（`tengu_amber_flint`）可能处于关闭状态。这是 Anthropic 的远程控制开关，即使你本地启用了环境变量，Anthropic 也可以从远端关闭 Agent Teams 功能。这是最容易被忽略的一层门控。

### 5.4 总结

&emsp;&emsp;Agent Teams 引入的 teammate 持久化基础（idle 与 active 循环、P2P 通信、共享 TaskList），为更复杂的多 Agent 工作流打下了基础。第四节课会讲的 fufan-cc-flow 多 Agent 工作流，正是建立在这些基础能力之上——当 Agent 之间的协作从「派活-收活」升级到「持久化协商」后，才有可能构建出真正的流水线式多 Agent 系统。

&emsp;&emsp;但这也是今天这节课需要诚实划界的地方：Agent Teams 是实验性功能，它的 API 和行为还在快速迭代中。你今天学到的源码结构和设计思想是稳定的——hub-and-spoke vs P2P mesh 的架构取舍、文件系统邮箱的零依赖设计、声明式依赖管理、有 leader 的 P2P——这些设计原则不会因为版本更新而失效。但具体的函数签名、行号、参数名可能会随版本变化，用的时候记得 `grep` 验证。

&emsp;&emsp;最后回到第三节课最后留下的那句话：「从今天起，你看任何一个 Agent，都会下意识地拿这把尺去量它。」

&emsp;&emsp;今天这节课做的事情，是把这把尺**往上拧了一格**。第三节课你学的是 Coordinator 模式下的四对张力——那是 hub-and-spoke 架构内的张力平衡。今天你看到了当 hub-and-spoke 碰到天花板后，Agent Teams 如何通过 P2P mesh + 共享 TaskList + 持久化 teammate 把「分治」的边界往外推了一步。这不是推翻旧的平衡，而是在更高的协作复杂度上找到了新的平衡点——**有 leader 的 P2P**。

&emsp;&emsp;以后你评估一个多 Agent 系统时，可以多问一个问题：「它的 Agent 之间是一次性的 dispatch 还是持久化的协作？如果是后者，它有没有一个像 TaskList 这样的共享状态来保证协调的可观测性？」能问出这个问题，说明你对多 Agent 协作的理解已经从「工具使用」升级到了「架构判断」。

> **【学完本节你已经掌握】**：你现在能把四对张力逐一映射到 Agent Teams 的具体机制上，能用「有 leader 的 P2P」一句话概括 Agent Teams 的核心设计取舍，也能讲清 feature flag 三层门控的设计意图（内部直通 / 外部双入口 / 远端 killswitch）。这 5 件产物——判断选型的能力、6 个工具的职能分工、TaskList 的结构差异、三种后端与身份系统、四对张力的升级映射——就是你从这节课带走的全部。